In [65]:
# python
import transformers, inspect, sys
print("transformers", transformers.__version__)
print("module file:", transformers.__file__)
import importlib
mod = importlib.import_module("transformers.generation")
print("generation module:", mod.__file__)
print("names in generation:", [n for n in dir(mod) if "Generation" in n or "generation" in n.lower()])
from typing import List, Dict, Any, Optional

transformers 4.57.3
module file: /Users/mattias/anaconda3/lib/python3.11/site-packages/transformers/__init__.py
generation module: /Users/mattias/anaconda3/lib/python3.11/site-packages/transformers/generation/__init__.py
names in generation: ['GenerationConfig', 'GenerationMixin', 'GenerationMode', 'TFGenerationMixin']


In [ ]:
from transformers import pipeline

# Initialize pipelines for various NLP tasks
# RoBERTa-based emotion classification trained on Reddit data
classifier = pipeline(task="text-classification", model="SamLowe/roberta-base-go_emotions", top_k=None)

# BART-based summarization model
summarizer = pipeline(task="summarization", model="facebook/bart-large-cnn")

# BERT-based mental health classification model for crisis detection
alarm = pipeline(task="text-classification", model="ourafla/mental-health-bert-finetuned", top_k=None)

# XLM-RoBERTa-based language detection model
lng_detect = pipeline(task="text-classification", model="papluca/xlm-roberta-base-language-detection")


Device set to use mps:0
Device set to use mps:0
Device set to use mps:0
Device set to use mps:0


[{'summary_text': '"I\'m trying not to judge myself for it. I know these dips happen," says the writer. "I\'m learning to treat myself gently on days like this," she says. "Even if it doesn\'t fully lift, it\'s okay" "I think I need more moments like that," she adds.'}]
[[{'label': 'optimism', 'score': 0.35791003704071045}, {'label': 'desire', 'score': 0.2260846048593521}, {'label': 'approval', 'score': 0.16971108317375183}, {'label': 'neutral', 'score': 0.16168662905693054}, {'label': 'realization', 'score': 0.06038009747862816}, {'label': 'caring', 'score': 0.043755996972322464}, {'label': 'disappointment', 'score': 0.01885879971086979}, {'label': 'relief', 'score': 0.01845538057386875}, {'label': 'joy', 'score': 0.018027516081929207}, {'label': 'admiration', 'score': 0.010631074197590351}, {'label': 'sadness', 'score': 0.008070033974945545}, {'label': 'annoyance', 'score': 0.007258594501763582}, {'label': 'disapproval', 'score': 0.006731878966093063}, {'label': 'pride', 'score': 0.0

### Clustering emotions from the go-emotions dataset, used in SamLowe's RoBERTa-based model.

In [ ]:
# --------------- EMOTION CLUSTERS (go_emotions → app schema) -----------------

# Raw labels used by SamLowe/roberta-base-go_emotions include (among others):
# admiration, amusement, anger, annoyance, approval, caring, confusion,
# curiosity, desire, disappointment, disapproval, disgust, embarrassment,
# excitement, fear, gratitude, grief, joy, love, nervousness, optimism,
# pride, realization, relief, remorse, sadness, surprise, neutral

EMOTION_CLUSTERS: Dict[str, List[str]] = {
    # “Negative” affect
    "sadness": ["sadness", "disappointment", "grief", "remorse"],
    "anxiety_fear": ["fear", "nervousness", "embarrassment"],
    "anger": ["anger", "annoyance", "disgust", "disapproval"],

    # “Positive” affect
    "joy": ["joy", "amusement", "excitement", "gratitude", "relief", "pride"],
    "love_caring": ["love", "caring", "admiration", "approval"],

    # Mixed / cognitive
    "hope_desire": ["optimism", "desire"],
    "confusion_uncertainty": ["confusion", "curiosity", "realization", "surprise"],

    # Baseline / no strong emotion
    "neutral": ["neutral"],
}

DEFAULT_EMOTION_THRESHOLD = 0.35  # Tunable threshold for multi-label selection


def aggregate_emotions(
    raw_emotion_outputs: List[Dict[str, Any]],
    threshold: float = DEFAULT_EMOTION_THRESHOLD
) -> Dict[str, Any]:
    """
    Takes the raw go_emotions output (list of {label, score}) and returns:
      - raw_sorted: raw emotions sorted by score
      - top_emotions: emotions whose score >= threshold
      - cluster_scores: max score per emotion cluster
      - cluster_ranking: clusters sorted by score desc
    """

    if not raw_emotion_outputs:
        return {
            "raw_sorted": [],
            "top_emotions": [],
            "cluster_scores": {},
            "cluster_ranking": [],
        }

    # Sort by descending score for consistency
    raw_sorted = sorted(raw_emotion_outputs, key=lambda e: e["score"], reverse=True)

    # Multi-label selection via threshold
    top_emotions = [e for e in raw_sorted if e["score"] >= threshold]

    # Map to clusters
    label_scores = {e["label"]: e["score"] for e in raw_sorted}
    cluster_scores: Dict[str, float] = {}

    for cluster_name, labels in EMOTION_CLUSTERS.items():
        scores = [label_scores.get(lbl, 0.0) for lbl in labels]
        if scores:
            cluster_scores[cluster_name] = max(scores)

    cluster_ranking = sorted(
        cluster_scores.items(), key=lambda kv: kv[1], reverse=True
    )

    return {
        "raw_sorted": raw_sorted,
        "top_emotions": top_emotions,
        "cluster_scores": cluster_scores,
        "cluster_ranking": cluster_ranking,
    }


### Crisis classifier adjustment

In [68]:

# You may need to adjust this label name depending on the model's actual outputs.
# Common ones: "suicidal", "suicide", "suicidal_thoughts".
SUICIDAL_LABEL_ID = ['1']
DEPRESSION_LABEL_ID = ['2']

DEFAULT_SUICIDAL_THRESHOLD = 0.5  # tuning knob
DEPRESSION_CRISIS_THRESHOLD = 0.9  # tuning knob


def assess_suicidal_risk(
    raw_risk_outputs: List[Dict[str, Any]],
    suicidal_threshold: float = DEFAULT_SUICIDAL_THRESHOLD,
    depression_threshold: float = DEPRESSION_CRISIS_THRESHOLD,
) -> Dict[str, Any]:

    if not raw_risk_outputs:
        return {
            "is_crisis": False,
            "reason": "none",
            "suicidal_score": None,
            "suicidal_label": None,
            "depression_score": None,
            "depression_label": None,
            "raw_risk_outputs": [],
        }

    suicidal_score: Optional[float] = None
    suicidal_label: Optional[str] = None
    depression_score: Optional[float] = None
    depression_label: Optional[str] = None

    for out in raw_risk_outputs:
        raw_label = str(out.get("label", ""))
        label_lower = raw_label.lower()
        score = float(out.get("score", 0.0))

        # Check suicidal labels (by name)
        if any(candidate in label_lower for candidate in SUICIDAL_LABEL_ID):
            if suicidal_score is None or score > suicidal_score:
                suicidal_score = score
                suicidal_label = raw_label

        # Check depression labels (by name)
        if any(candidate in label_lower for candidate in DEPRESSION_LABEL_ID):
            if depression_score is None or score > depression_score:
                depression_score = score
                depression_label = raw_label

        # Also check depression by ID (LABEL_2 / "2")
        if raw_label in DEPRESSION_LABEL_ID:
            if depression_score is None or score > depression_score:
                depression_score = score
                depression_label = raw_label

    # Crisis logic
    suicidal_crisis = suicidal_score is not None and suicidal_score >= suicidal_threshold
    depression_crisis = depression_score is not None and depression_score >= depression_threshold

    if suicidal_crisis:
        is_crisis = True
        reason = "suicidal"
    elif depression_crisis:
        is_crisis = True
        reason = "severe_depression"
    else:
        is_crisis = False
        reason = "none"

    return {
        "is_crisis": is_crisis,
        "reason": reason,
        "suicidal_score": suicidal_score,
        "suicidal_label": suicidal_label,
        "depression_score": depression_score,
        "depression_label": depression_label,
        "raw_risk_outputs": raw_risk_outputs,
    }

In [45]:
import re
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

from transformers import AutoTokenizer


@dataclass
class PreprocessConfig:
    # Basic cleaning
    lowercase: bool = False
    strip_whitespace: bool = True
    collapse_whitespace: bool = True
    normalize_repeated_chars: bool = True
    max_repeated_chars: int = 3   # "soooo" -> "sooo"

    # Chunking for models with context limits (e.g. BART)
    use_token_chunking: bool = True
    max_tokens: int = 512         # per chunk, conservatively under 1024 for Bart
    overlap_tokens: int = 50      # overlap between chunks to keep context


class JournalPreprocessor:
    def __init__(
        self,
        bart_model_name: str = "facebook/bart-large-cnn",
        config: Optional[PreprocessConfig] = None
    ):
        self.config = config or PreprocessConfig()
        # Tokenizer used for chunking (BART tokenizer so we respect its limits)
        self.tokenizer = AutoTokenizer.from_pretrained(bart_model_name)

    # ----- PUBLIC API -----
    def preprocess_for_all(self, text: str) -> Dict[str, Any]:
        """
        Returns:
            {
                'clean_text': str,            # cleaned text for emotion / risk models
                'chunks_for_summarizer': [str]  # list of chunks for BART
            }
        """
        clean = self._basic_clean(text)

        if self.config.use_token_chunking:
            chunks = self._chunk_by_tokens(clean)
        else:
            chunks = [clean]

        return {
            "clean_text": clean,
            "chunks_for_summarizer": chunks
        }

    # ----- BASIC CLEANING -----
    def _basic_clean(self, text: str) -> str:
        if not isinstance(text, str):
            text = str(text)

        # Normalize newlines
        text = text.replace("\r\n", "\n").replace("\r", "\n")

        if self.config.strip_whitespace:
            text = text.strip()

        if self.config.collapse_whitespace:
            text = re.sub(r"\s+", " ", text)

        if self.config.normalize_repeated_chars:
            # Limit repeated characters: "soooo" -> "sooo"
            # Avoid breaking emojis by restricting to ASCII letters
            n = self.config.max_repeated_chars
            text = re.sub(r"([A-Za-z])\1{" + str(n) + r",}", r"\1" * n, text)

        if self.config.lowercase:
            text = text.lower()

        return text

    # ----- CHUNKING -----
    def _chunk_by_tokens(self, text: str) -> List[str]:
        """
        Split text into overlapping token chunks for models like BART.

        Returns list of decoded text chunks.
        """
        if not text:
            return [""]

        encoding = self.tokenizer(
            text,
            add_special_tokens=False,
            return_attention_mask=False,
            return_tensors=None
        )
        input_ids = encoding["input_ids"]

        max_len = self.config.max_tokens
        overlap = self.config.overlap_tokens

        if len(input_ids) <= max_len:
            return [text]

        chunks = []
        start = 0

        while start < len(input_ids):
            end = start + max_len
            chunk_ids = input_ids[start:end]
            chunk_text = self.tokenizer.decode(chunk_ids, skip_special_tokens=True)
            chunks.append(chunk_text)

            if end >= len(input_ids):
                break

            # Move forward with overlap
            start = end - overlap

        return chunks



In [ ]:
# Init preprocessor once
preprocessor = JournalPreprocessor(
    bart_model_name="facebook/bart-large-cnn",
    config=PreprocessConfig(
        lowercase=False,        # keep original case for readability
        max_tokens=512,         # safe chunk size for BART
        overlap_tokens=50
    )
)

def process_journal_entry(raw_text: str):
    # 0) Preprocess
    prep = preprocessor.preprocess_for_all(raw_text)
    clean_text = prep["clean_text"]
    chunks = prep["chunks_for_summarizer"]

    # 1) Emotions on full cleaned text
    emotion_outputs = classifier(clean_text)[0]  # go_emotions
    emotion_analysis = aggregate_emotions(emotion_outputs)

    # 2) Full risk outputs (RAW)
    raw_risk_outputs = alarm(clean_text)[0]

    # 3) Suicidal-only assessment (debug-friendly)
    suicidal_assessment = assess_suicidal_risk(raw_risk_outputs)

    # 4) Summaries on chunks
    chunk_summaries = []
    for chunk in chunks:
        if not chunk.strip():
            continue
        summary = summarizer(
            chunk,
            max_length=120,
            min_length=40,
            do_sample=False
        )[0]["summary_text"]
        chunk_summaries.append(summary)

    # Combine summaries if needed
    if len(chunk_summaries) == 1:
        final_summary = chunk_summaries[0]
    elif len(chunk_summaries) > 1:
        combined = " ".join(chunk_summaries)
        final_summary = summarizer(
            combined,
            max_length=150,
            min_length=60,
            do_sample=False
        )[0]["summary_text"]
    else:
        final_summary = ""

    # ---- Return all diagnostic information ----
    return {
        "clean_text": clean_text,

        # Emotion analysis
        "emotion_outputs_raw": emotion_analysis["raw_sorted"],
        "emotion_top_labels": emotion_analysis["top_emotions"],
        "emotion_cluster_scores": emotion_analysis["cluster_scores"],
        "emotion_cluster_ranking": emotion_analysis["cluster_ranking"],

        # FULL ALARM RAW OUTPUTS (for debugging)
        "risk_outputs_raw": raw_risk_outputs,

        # Suicidal-only filtered assessment
        "suicidal_is_crisis": suicidal_assessment["is_crisis"],
        "suicidal_score": suicidal_assessment["suicidal_score"],
        "suicidal_label": suicidal_assessment["suicidal_label"],

        # Summaries
        "summary": final_summary,
        "chunk_summaries": chunk_summaries,
    }


In [61]:
# Test entries

calibration_corpus = "I don't remember the last time I felt truly happy."

corpus = "I woke up feeling heavier than usual today. Not sad exactly—more like my thoughts were moving through syrup. " \
"I tried to ignore it at first, but it followed me from the kitchen to my desk. I'm trying not to judge myself for it. " \
"I know these dips happen. I took a short walk at lunch, and it helped more than I expected. " \
"The air was cool, and the sunlight felt grounding. For a few minutes, my mind quieted down enough for me to breathe. " \
"I think I need more moments like that—small pauses to remind myself I'm not just what I'm feeling in the moment. " \
"Tonight I'm planning to make tea and read something comforting. I'm hoping that giving myself a bit of softness will help reset my mood. " \
"Even if it doesn't fully lift, it's okay. I'm learning to treat myself gently on days like this." 

other_lang = """
Je me suis encore mal reposé. Je me sens tellement épuisé et engourdi.
J'ai peur de faire des erreurs au travail, et je n'arrive pas à arrêter de trop réfléchir à tout.
Faire une promenade m'a un peu aidé, mais je me sens toujours lourd.
"""

entry = """
Slept terribly again. I feel sooo exhausted and numb.
I'm scared I'm going to mess up at work, and I can't stop overthinking everything.
Going for a walk helped a bit, but I still feel heavy.
"""

alarm_test = """I can't take this anymore. Life feels unbearable, and I just want the pain to end."""

alarm_test_2 = """Sometimes I think about disappearing."""

In [ ]:

# Printing results
def print_results(text_entry: str):
    if lng_detect([text_entry])[0]['label'] != 'en':
        print("Input text is not in English. As of right now, only English is supported.")
        sys.exit(0)
    else:
        result = process_journal_entry(text_entry)
        print("CLEAN TEXT:\n", result["clean_text"])
        print("\nSUMMARY:\n", result["summary"])

        print("\nTOP EMOTION LABELS (>= threshold):")
        for emo in result["emotion_top_labels"]:
            print(f"  - {emo['label']}: {emo['score']:.3f}")

        print("\nEMOTION CLUSTERS DEBUG (sorted):")
        for cluster, score in result["emotion_cluster_ranking"]:
            print(f"  - {cluster}: {score:.3f}")

        print("\n=== RAW RISK OUTPUTS FROM MODEL ===")
        for r in result["risk_outputs_raw"]:
            print(f"{r['label']:20s} : {r['score']:.4f}")

        print("\nSuicidal label detected:", result["suicidal_label"])
        print("Suicidal score:", result["suicidal_score"])
        print("Crisis flagged:", result["suicidal_is_crisis"])
    
            
        # ---- CRISIS HANDLING ----
        if result["suicidal_is_crisis"]:
            print("This entry appears to express suicidal thoughts.")
            print("Suicidal score:", f"{result['suicidal_score']:.3f}" if result["suicidal_score"] is not None else "N/A")

            # Example: crisis resources message (you'd adapt this to regions / languages)
            print(
                "\nIf you are in immediate danger or thinking about harming yourself, "
                "please reach out to a crisis hotline right now. 113 is free and available 24/7.\n\n"
                "If you are able, consider also talking to a trusted person or a mental health professional.\n\n"
                "This app is not a replacement for these services."
            )
        else:
            print("\nNo crisis-level suicidal risk detected (based on current threshold).")

print_results(alarm_test)

Your max_length is set to 120, but your input_length is only 22. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=11)


CLEAN TEXT:
 I can't take this anymore. Life feels unbearable, and I just want the pain to end.

SUMMARY:
 "I can't take this anymore. Life feels unbearable, and I just want the pain to end," she says. "I just want to end. I can't do it anymore"

TOP EMOTION LABELS (>= threshold):
  - sadness: 0.624
  - desire: 0.432

EMOTION CLUSTERS (sorted):
  - sadness: 0.624
  - hope_desire: 0.432
  - anger: 0.043
  - neutral: 0.042
  - love_caring: 0.018
  - confusion_uncertainty: 0.011
  - anxiety_fear: 0.009
  - joy: 0.006

=== RAW RISK OUTPUTS FROM MODEL ===
LABEL_1              : 0.9904
LABEL_3              : 0.0080
LABEL_0              : 0.0014
LABEL_2              : 0.0002

Suicidal label detected: LABEL_1
Suicidal score: 0.9903544187545776
Crisis flagged: True
This entry appears to express suicidal thoughts.
Suicidal score: 0.990

If you are in immediate danger or thinking about harming yourself, please reach out to a crisis hotline right now. 113 is free and available 24/7.

If you are ab

Kaggle link for the dataset used below: https://www.kaggle.com/datasets/atharvjairath/empathetic-dialogues-facebook-ai?resource=download 

In [ ]:
import pandas as pd


# Empathetic dialogues dataset, labeled data
df = pd.read_csv('emotion-emotion_69k.csv')
df.drop(['Unnamed: 0', 'empathetic_dialogues', 'labels', 'Unnamed: 5', 'Unnamed: 6'], axis=1, inplace=True)

df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
print(df)

                                               Situation       emotion
0      I remember going to the fireworks with my best...   sentimental
1                           i used to scare for darkness        afraid
2      I showed a guy how to run a good bead in weldi...         proud
3                   I have always been loyal to my wife.      faithful
4      A recent job interview that I had made me feel...     terrified
...                                                  ...           ...
19299  I was watching professional rodeo the other da...     impressed
19300  I am waiting to see if I pass my graduate exam...  anticipating
19301  My house burned down and I had to rescue my fa...        afraid
19302  I found some pictures of my grandma in the att...   sentimental
19303  I woke up this morning to my wife telling me s...     surprised

[19304 rows x 2 columns]


In [26]:
# python
print(df['emotion'].value_counts())
print(df['emotion'].isna().sum(), "missing labels")

emotion
surprised                                                                                                                                                                                         997
excited                                                                                                                                                                                           739
angry                                                                                                                                                                                             686
proud                                                                                                                                                                                             670
sad                                                                                                                                                                                               663
an